# NT/GUE data preview and GROVER smoke test

This notebook previews one NT task and one GUE task, prints class ratios, then runs the existing `glm_finetune.train` smoke-test path with the local GROVER checkpoint.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_ROOT = Path("/dataset/zjn_zjj/DLM/GLM_Benchmark")
FINETUNE_ROOT = REPO_ROOT / "Exp1_Public-dataset-bench"
SRC_ROOT = FINETUNE_ROOT / "src"
DATA_ROOT = Path("/home/zhangjiajing/2026_Graduation_Project/Data")
MODEL_ROOT = Path("/dataset/zjn_zjj/DLM/GFM_model_files")
OUTPUT_ROOT = FINETUNE_ROOT / "outputs" / "notebook_smoke"

sys.path.insert(0, str(SRC_ROOT))
print(f"data_root={DATA_ROOT}")
print(f"model_root={MODEL_ROOT}")
print(f"output_root={OUTPUT_ROOT}")


data_root=/home/zhangjiajing/2026_Graduation_Project/Data
model_root=/dataset/zjn_zjj/DLM/GFM_model_files
output_root=/dataset/zjn_zjj/DLM/GLM_Benchmark/Exp1_Public-dataset-bench/finetune/outputs/notebook_smoke


In [2]:
import pandas as pd
from glm_finetune.datasets import load_task

def preview_task(dataset_name: str, task_name: str):
    split = load_task(DATA_ROOT, dataset_name, task_name, seed=42)
    print(f"{dataset_name}/{task_name}")
    print(f"train={len(split.train)} validation={len(split.validation)} test={len(split.test)}")
    display(split.train.head())

    counts = split.train["label"].value_counts().sort_index()
    ratios = counts / counts.sum()
    summary = pd.DataFrame({"count": counts, "ratio": ratios})
    summary.index = [split.id2label[int(label)] for label in summary.index]
    summary.index.name = "label"
    display(summary)
    return split

nt_split = preview_task("NT", "enhancers")
gue_split = preview_task("GUE", "EMP/H3")


NT/enhancers
train=27000 validation=3000 test=3000


,sequence,label
0,GTCCCTGCCCTGTCCGGAGCCCTCCCTCAGCCCTGGGCTCACCTGG...,1
1,CCTGTTACTGCTGTCTGCAAGGTTGCAACTAATATTCATCATCTTC...,0
2,AGTCAAATTTATCCCTGGAAAAAAACCAAACAAACATAGCTTGACT...,0
3,AATTGCCTGATCGCTCTGGCCAGGACTTCCAGTACTCTGTTGAATA...,1
4,GGAAATATAATTGTCATAATTATATGTGTTACACATTATGATAGTA...,1


,count,ratio
label,,
0,13495,0.499815
1,13505,0.500185


GUE/EMP/H3
train=11971 validation=1497 test=1497


,sequence,label
0,CTGAAATTTTTTTCGTTTCGCCACCAAGAAAGGCTAATTGAATACA...,1
1,ATATGCTCGAAAATGTAGCTATATTTCACGGATGAATAACTCGTAA...,0
2,CAACGGTACCCAATGGTTCCATCTCTAAGTCGTTAGGGCCAATCGT...,1
3,AACGAAGTCTAAATGGATCGAAGAAAGATTTCAAGAAAAGGTAAGT...,0
4,TGAGCAAATTTCCTTCCAACTTAGATGAACCCAGTAATTTTATTAA...,0


,count,ratio
label,,
0,5821,0.486258
1,6150,0.513742


The smoke test below reuses `glm_finetune.train`: it loads data, tokenizer, and model, builds one tiny batch, runs a forward pass, and exits before full training.


In [3]:
env = os.environ.copy()
env["PYTHONPATH"] = f"{SRC_ROOT}:{env.get('PYTHONPATH', '')}"
env["CUDA_VISIBLE_DEVICES"] = env.get("CUDA_VISIBLE_DEVICES", "0")

cmd = [
    sys.executable,
    "-m",
    "glm_finetune.train",
    "--model",
    "grover",
    "--dataset",
    "NT",
    "--task",
    "enhancers",
    "--data-root",
    str(DATA_ROOT),
    "--model-root",
    str(MODEL_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--smoke-test",
    "--max-train-samples",
    "4",
    "--max-eval-samples",
    "4",
    "--token-length-sample-size",
    "64",
]

print(" ".join(cmd))
subprocess.run(cmd, check=True, env=env)


/home/zhangjiajing/miniforge3/envs/glm_hf/bin/python -m glm_finetune.train --model grover --dataset NT --task enhancers --data-root /home/zhangjiajing/2026_Graduation_Project/Data --model-root /dataset/zjn_zjj/DLM/GFM_model_files --output-root /dataset/zjn_zjj/DLM/GLM_Benchmark/Exp1_Public-dataset-bench/finetune/outputs/notebook_smoke --smoke-test --max-train-samples 4 --max-eval-samples 4 --token-length-sample-size 64


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /dataset/zjn_zjj/DLM/GFM_model_files/GROVER and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SMOKE_OK grover NT/enhancers max_length=106


CompletedProcess(args=['/home/zhangjiajing/miniforge3/envs/glm_hf/bin/python', '-m', 'glm_finetune.train', '--model', 'grover', '--dataset', 'NT', '--task', 'enhancers', '--data-root', '/home/zhangjiajing/2026_Graduation_Project/Data', '--model-root', '/dataset/zjn_zjj/DLM/GFM_model_files', '--output-root', '/dataset/zjn_zjj/DLM/GLM_Benchmark/Exp1_Public-dataset-bench/finetune/outputs/notebook_smoke', '--smoke-test', '--max-train-samples', '4', '--max-eval-samples', '4', '--token-length-sample-size', '64'], returncode=0)